## Sample OSM Polygons and Get (Uncropped) EO Images

### This notebook samples building polygons from a predefined list of cities using OpenStreetMap (OSM) via the osmnx library. It extracts the center coordinates of a random subset of these buildings and automates the downloading of satellite imagery using the Google Maps Static API, utilizing robust retry and logging logic.

In [ ]:
import os
import time
import requests
import pandas as pd
import osmnx as ox
import warnings
import json

# Suppress GeoPandas UserWarning regarding centroid calculation on geographic CRS
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")

# === CONFIGURATION AND API SETUP ===

# Load Google Maps Static API key
with open('../resources/keys.json', 'r') as file:
    data = json.load(file)
# Get the value
API_KEY = data["google_static_maps_api_key"]
if API_KEY=='placeholder':
    raise ValueError('Cannot use placeholder for Google Static Maps API key.')

# Output directory where the downloaded satellite images will be stored
OUT_DIR = "osm_sampled_roofs" 

# CSV file to record metadata for any downloads that fail after multiple attempts
FAILED_CSV = "failed_downloads.csv" 

# CSV file to record metadata for any successful downloads
SUCCESS_CSV = "test_add_cities.csv" 

# List of areas to sample from (ensure the naming is clear for OSM Nominatim geocoder)
# See https://nominatim.openstreetmap.org/ui/
CITIES = [
    "Chicago, South Chicago Township, Cook County, Illinois, United States",
    "Almaty, Kazakhstan",
    "Port Moresby, National Capital District, Southern Region, 111, Papua New Guinea"
]

# Number of building polygons to randomly sample per city to avoid exhausting API limits
SAMPLES_PER_CITY = 10

# Ensure the output directory exists on the filesystem
os.makedirs(OUT_DIR, exist_ok=True) 

In [2]:
# === HELPER FUNCTIONS ===

def sample_buildings_from_cities(cities, n_samples, distance_m=None):
    """
    Fetches building footprints from OSM for a list of cities, samples them, 
    and extracts their centroid coordinates.
    """
    building_records = []
    
    for city in cities:
        print(f"Fetching building footprints for: {city}...")
        try:
            if distance_m is not None:
                # Geocode the city to get a center point
                center_point = ox.geocoder.geocode(city) # Returns (lat, lon)

                # Fetch features within a small radius (e.g., 100 meters)
                gdf = ox.features_from_point(center_point, tags={'building': True}, dist=distance_m)

            else:
                # Fetch building features from OSM
                gdf = ox.features_from_place(city, tags={'building': True})
                # Filter for Polygons and MultiPolygons to ensure we have valid building footprints
                gdf = gdf[gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])]
                # Sample a subset to avoid massive API bills
                if len(gdf) > n_samples:
                    gdf = gdf.sample(n=n_samples, random_state=42)

            # Apply your filters and limit
            gdf = gdf[gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])]
            if len(gdf) > n_samples:
                gdf = gdf.iloc[:n_samples]
            
            # Extract centroids (lat/lon) for the Google Static Maps API center point
            for idx, row in gdf.iterrows():
                centroid = row.geometry.centroid
                
                # Create a clean prefix for the filename based on the city name
                city_prefix = city.split(',')[0].replace(' ', '_').lower()
                
                # Handle potential MultiIndex from osmnx (element_type, osmid)
                osmid = idx[1] if isinstance(idx, tuple) else idx
                
                building_records.append({
                    "filename": f"{city_prefix}_{osmid}.png",
                    "latitude": centroid.y,
                    "longitude": centroid.x,
                    "city": city
                })
            print(f"Successfully sampled {len(gdf)} buildings from {city}.")
            
        except Exception as e:
            print(f"Error fetching OSM data for {city}: {e}")
            
    return pd.DataFrame(building_records)

def get_image(lat, lon):
    """
    Constructs and sends a request to the Google Static Maps API for a satellite 
    image centered at the provided coordinates.
    """
    url = (
        "https://maps.googleapis.com/maps/api/staticmap"
        f"?center={lat},{lon}"
        "&zoom=20"            # High-resolution building views
        "&size=256x256"       # Request 256x256 pixel tiles
        "&maptype=satellite"  # Use satellite imagery layer
        f"&key={API_KEY}"
    )
    return session.get(url, timeout=30)

In [ ]:
# Initialize a persistent requests session to improve performance
session = requests.Session()

# === STEP 1: GENERATE DOWNLOAD QUEUE FROM OSM ===

print("=== Starting OSM Building Sampling ===")
df = sample_buildings_from_cities(CITIES, SAMPLES_PER_CITY, distance_m=100)

if df.empty:
    print("No buildings successfully sampled. Exiting.")
    exit()

print(f"\nTotal buildings to process: {len(df)}")

# === STEP 2: MAIN DOWNLOAD LOOP ===

print("\n=== Starting Google Static Maps Downloads ===")
failed_rows = [] 
success_records = [] # Track successful downloads for the metadata CSV

for i, row in df.iterrows():
    filename = row["filename"]
    lat = row["latitude"]
    lon = row["longitude"]
    city = row.get("city", "unknown") # Ensure your sampling function provides the city name

    # Define the full destination path for the current image
    out_path = os.path.join(OUT_DIR, filename) 

    # If the file already exists, we count it as a success for the metadata CSV
    if os.path.exists(out_path):
        success_records.append(row)
        continue

    # Skip download if coordinates are missing or invalid
    if pd.isna(lat) or pd.isna(lon): 
        print(f"Bad coordinates: {filename} | lat={lat} lon={lon}")
        failed_rows.append({
            "filename": filename,
            "latitude": lat,
            "longitude": lon,
            "reason": "bad_coordinates"
        })
        continue

    success = False 

    # Attempt to download the image up to 5 times
    for attempt in range(1, 6): 
        try:
            response = get_image(lat, lon)
            content_type = response.headers.get("Content-Type", "")
            
            if response.status_code == 200 and "image" in content_type.lower(): 
                with open(out_path, "wb") as f:
                    f.write(response.content)
                success = True
                break
            
        except Exception as e:
            print(f"Attempt {attempt}/5 error for {filename}: {e}")

        time.sleep(2 * attempt) 

    if success:
        # Add to success records to build the metadata CSV
        success_records.append(row)
    else:
        failed_rows.append({
            "filename": filename,
            "latitude": lat,
            "longitude": lon,
            "reason": "request_failed"
        })

    if (i + 1) % 10 == 0: 
        print(f"Processed {i + 1}/{len(df)}")

    time.sleep(0.2)

# === STEP 3: FINALIZATION & METADATA EXPORT ===

# 1. Save Failed Downloads
if failed_rows: 
    pd.DataFrame(failed_rows).to_csv(FAILED_CSV, index=False)
    print(f"\nSaved {len(failed_rows)} failed rows to {FAILED_CSV}")

# 2. Save Success Records (Modified to include EVERY polygon)
if success_records:
    success_df = pd.DataFrame(success_records)
    
    # We construct the dataframe using every row in success_records.
    # Included 'filename' so you can map these coordinates back to the actual images.
    compliant_df = pd.DataFrame({
        'filename': success_df['filename'],
        'city': success_df['city'] if 'city' in success_df else 'unknown',
        'latitude': success_df['latitude'],
        'longitude': success_df['longitude'],
    })
    
    # REMOVED: compliant_df.drop_duplicates(subset=['City'])
    # This ensures every sampled building is preserved in the CSV.
    
    # Use utf-8 or latin1 depending on your OS/Excel requirements
    compliant_df.to_csv(SUCCESS_CSV, index=False, encoding='utf-8')
    
    print(f"✅ Full metadata CSV saved to: {SUCCESS_CSV}")
    print(f"Total entries: {len(compliant_df)} (one for every sampled building)")
else:
    print("\nNo images were successfully processed.")